# 3. Modelos benchmark

:::{admonition} Qué responde este capítulo
:class: tip
Cuál es la mejor combinación (balanceo y optimizador) de cada modelo, cómo se eligió, cuánto rinde en una validación
repetida que no participó en la elección y cómo se reentrenó y serializó.
:::

## 3.1 Criterio de selección

Del universo del E3 (8 modelos × 4 técnicas de balanceo × 4 métodos de optimización = 128 combinaciones, cada una
con validación anidada kNNDM de 5 × 3 pliegues y 12 evaluaciones), para cada uno de los **7 modelos de la guía** se
toma la combinación con mayor **F1 macro medio del bucle externo**. Se agregan el XGBoost geográfico (la propuesta
del E2/E3, como referencia) y dos métodos recientes del estado del arte, **LightGBM** {cite}`ke2017` y
**CatBoost** {cite}`prokhorenkova2018`, con pesos de clase y búsqueda bayesiana, el mismo presupuesto y las mismas
particiones.

La mejor combinación global (Random Forest con SMOTE y algoritmo genético, 0.658) está en el grupo de cabeza de
Friedman → Nemenyi del E3 (a 1.02 rangos del mejor, con diferencia crítica 1.86): el criterio es estadístico y la
elección es consistente con la comparación múltiple.

## 3.2 Mejores combinaciones en la validación repetida 5 × 5

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().resolve().parents[1]
T = pd.read_csv(RAIZ / "resultados" / "cv5x5" / "f1_por_pliegue_v2.csv")
meta = json.loads((RAIZ / "artifacts" / "metadata.json").read_text(encoding="utf-8"))
filas = []
for nombre, m in meta["modelos"].items():
    if nombre == "EPL (original)":
        continue
    col = T[nombre]
    filas.append({"modelo": nombre, "grupo": m["grupo"], "balanceo": m["balanceo"], "optimizador": m["metodo"],
                  "F1 rep. 0 (E3)": round(col[T.rep == 0].mean(), 3), "F1 reps. 1–4": round(col[T.rep > 0].mean(), 3),
                  "F1 25 pliegues": f"{col.mean():.3f} ± {col.std(ddof=1):.3f}",
                  "inferencia (ms/viv.)": round(m["ms_inferencia_por_vivienda"], 3)})
tabla = pd.DataFrame(filas).sort_values("F1 reps. 1–4", ascending=False)
tabla.style.hide(axis="index")

modelo,grupo,balanceo,optimizador,F1 rep. 0 (E3),F1 reps. 1–4,F1 25 pliegues,inferencia (ms/viv.)
CatBoost,reciente,class_weight,Bayesiana,0.648000,0.595000,0.605 ± 0.103,0.019000
Random Forest,guia,smote,Genética,0.658000,0.592000,0.605 ± 0.097,0.057000
XGBoost,guia,smote,Random,0.656000,0.584000,0.599 ± 0.107,0.072000
Logística L1/L2,guia,ninguno,Bayesiana,0.607000,0.583000,0.588 ± 0.089,0.016000
XGBoost geográfico,referencia,class_weight,Bayesiana,0.656000,0.569000,0.586 ± 0.108,0.072000
LightGBM,reciente,class_weight,Bayesiana,0.654000,0.567000,0.584 ± 0.129,0.420000
SVM,guia,class_weight,Grid,0.617000,0.564000,0.575 ± 0.091,0.037000
KNN,guia,class_weight,Bayesiana,0.600000,0.551000,0.561 ± 0.099,0.773000
Árbol de decisión,guia,smote,Grid,0.599000,0.497000,0.518 ± 0.099,0.016000
Naive Bayes,guia,smote,Bayesiana,0.452000,0.395000,0.406 ± 0.077,0.017000


**Lectura.** La repetición 0 son los mismos pliegues con que el E3 eligió las combinaciones; las repeticiones 1–4
son particiones kNNDM nuevas. **Todos los modelos bajan en las particiones nuevas** (el Random Forest de 0.658 a
0.592): es el sesgo de selección de haber elegido y reportado con los mismos pliegues (fila 17 del capítulo 1). El
orden se mantiene en la cabeza: Random Forest, CatBoost y XGBoost quedan a menos de 0.01 entre sí, y la logística y
el XGBoost geográfico detrás. Naive Bayes queda último, como en el E3.

## 3.3 Reentrenamiento final y serialización

Una vez fijada la combinación, el **mismo procedimiento** que se validó (búsqueda con el mismo método y 12
evaluaciones, usando los 5 pliegues externos de la repetición 0 como validación) se aplica a toda la submuestra
de entrenamiento y el pipeline completo —preprocesamiento del E1 → [SMOTE] → modelo— se reentrena con las 10 001
viviendas y se guarda con ``joblib.dump(..., compress=3)`` en ``artifacts/``. Para los 7 modelos de la guía y el
geográfico se reutilizan los hiperparámetros de esa búsqueda del E3; LightGBM, CatBoost y el modelo original se
buscan aquí. Junto a cada modelo, ``artifacts/metadata.json`` guarda versiones de las librerías, semilla,
hiperparámetros, esquema de columnas (tipo y rango válido), métricas del bucle externo y tiempos.

In [2]:
pd.DataFrame([{"modelo": n, "archivo": m["archivo"], "MB": m["tamano_MB"], "entrenamiento (s)": m["seg_entrenamiento"],
               "ida y vuelta idéntica": m["ida_y_vuelta_identica"], "hiperparámetros": json.dumps(m["params"])[:120]}
              for n, m in meta["modelos"].items()]).style.hide(axis="index")

modelo,archivo,MB,entrenamiento (s),ida y vuelta idéntica,hiperparámetros
KNN,knn.joblib,0.810000,0.200000,True,"{""n_neighbors"": 54, ""weights"": ""distance"", ""p"": 1}"
Naive Bayes,naive_bayes.joblib,0.040000,0.500000,True,"{""var_smoothing"": 0.03214719348281692}"
Logística L1/L2,logistica_l1_l2.joblib,0.010000,24.400000,True,"{""penalty"": ""l1"", ""C"": 0.47367173240839705}"
Árbol de decisión,arbol_de_decision.joblib,0.060000,1.300000,True,"{""max_depth"": 10, ""min_samples_leaf"": 5, ""criterion"": ""entropy""}"
Random Forest,random_forest.joblib,8.150000,16.700000,True,"{""n_estimators"": 141, ""max_depth"": 21, ""max_features"": 0.3, ""min_samples_leaf"": 2}"
XGBoost,xgboost.joblib,0.750000,10.400000,True,"{""n_estimators"": 202, ""learning_rate"": 0.04696344667875392, ""max_depth"": 5, ""subsample"": 0.6175215063148914, ""colsample_"
SVM,svm.joblib,0.730000,11.500000,True,"{""gamma"": 0.03, ""n_components"": 300, ""C"": 10.0}"
XGBoost geográfico,xgboost_geografico.joblib,2.420000,11.300000,True,"{""n_estimators"": 186, ""learning_rate"": 0.04386723024324426, ""max_depth"": 8, ""frac_vecinos"": 0.1233959246959953, ""alpha"":"
LightGBM,lightgbm.joblib,5.610000,11.500000,True,"{""n_estimators"": 318, ""learning_rate"": 0.16515182865193914, ""num_leaves"": 61, ""min_child_samples"": 5, ""subsample"": 0.970"
CatBoost,catboost.joblib,0.160000,7.200000,True,"{""iterations"": 230, ""learning_rate"": 0.15317709628184267, ""depth"": 4, ""l2_leaf_reg"": 9.330606024425668}"


**Prueba de ida y vuelta.** Cada archivo se vuelve a cargar y sus probabilidades sobre 2 000 viviendas fijas del test
se comparan con las del modelo en memoria: son **idénticas** en todos (``tests/test_artefactos.py`` lo repite en
cada ejecución de las pruebas). Los artefactos pesan entre 0.01 y 10.5 MB; el mayor es el modelo original, que
guarda sus 7 expertos. Ninguno exige Git LFS ni reducir árboles.